In [ ]:
from matplotlib import pyplot as plt
import tifffile
import numpy as np
from matplotlib.lines import Line2D
from cellpose import models
from cellpose.io import imread
from pathlib import Path
from cellpose.utils import masks_to_outlines
from scipy.ndimage import binary_dilation


In [ ]:
BASE_DIR   = Path(".")
TILES_DIR  = Path('data/tiles_sub')
# TILES_DIR = Path('data/tiles_sub')
MODEL_PATH = BASE_DIR / "models" / "Halo"

In [ ]:
tile_id = "tile_0013_base"

img = imread(TILES_DIR / "human_pancreas" / 'test' / 'images' / f"{tile_id}.tiff")
true_masks = np.load(TILES_DIR / "human_pancreas" / 'test' / 'masks' / f'{tile_id}.npy')
nuclei_masks = np.load(TILES_DIR / "human_pancreas" / 'test' / 'nuclei_mask' / f'{tile_id}.npy')
cp_masks =  np.load(TILES_DIR / "human_pancreas" / 'test' / 'mask_cpsam' / f'{tile_id}.npy')
xr_masks = np.load(TILES_DIR / 'human_pancreas' / 'test' / 'mask_xr' / f'{tile_id}.npy')
spot = np.load(TILES_DIR / 'human_pancreas' / 'test' / 'spots' / f'{tile_id}.npz')

true_outline = masks_to_outlines(true_masks)
cp_outline = masks_to_outlines(cp_masks)
xr_outline = masks_to_outlines(xr_masks)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import binary_dilation
from cellpose.utils import masks_to_outlines

# ============================================================
# Helpers
# ============================================================
def norm01(a):
    a = a.astype(float)
    mn, mx = a.min(), a.max()
    if mx > mn:
        return (a - mn) / (mx - mn)
    return np.zeros_like(a, dtype=float)

def crop_box_from_center(base_img, cx, cy, roi_w, roi_h):
    H, W = base_img.shape
    x0 = max(0, int(cx - roi_w // 2))
    x1 = min(W, int(cx + roi_w // 2))
    y0 = max(0, int(cy - roi_h // 2))
    y1 = min(H, int(cy + roi_h // 2))
    return x0, x1, y0, y1

def crop(a, x0, x1, y0, y1):
    return a[y0:y1, x0:x1]

def remove_instances_touching_crop_border(lbl):
    """
    lbl: 2D label image (0 background, 1..N instances)
    Return a copy where any instance touching the crop border is removed (set to 0).
    """
    if lbl.ndim != 2:
        raise ValueError("Label image must be 2D")

    H, W = lbl.shape
    if H == 0 or W == 0:
        return lbl.copy()

    border = np.concatenate([
        lbl[0, :],          # top
        lbl[-1, :],         # bottom
        lbl[:, 0],          # left
        lbl[:, -1],         # right
    ])
    bad = np.unique(border)
    bad = bad[bad != 0]  # ignore background

    if bad.size == 0:
        return lbl.copy()

    out = lbl.copy()
    # remove all pixels belonging to those border-touching labels
    out[np.isin(out, bad)] = 0
    return out

def make_base_rgb(base_crop):
    rgb = np.zeros((base_crop.shape[0], base_crop.shape[1], 3), dtype=float)
    rgb[..., 2] = base_crop  # BLUE
    return np.clip(rgb, 0, 1)

def split_spots_by_mask(mask_crop, sx, sy):
    inside = mask_crop[sy, sx] > 0
    return (sx[inside], sy[inside]), (sx[~inside], sy[~inside])

# ============================================================
# Normalize channel 0 ONLY (BLUE background)
# ============================================================
base_img = norm01(img[..., 0])

# ============================================================
# Crop config (selected cropped region)
# ============================================================
cx, cy = 256, 256
roi_w, roi_h = 512, 512

x0, x1, y0, y1 = crop_box_from_center(base_img, cx, cy, roi_w, roi_h)

base_crop = crop(base_img, x0, x1, y0, y1)
Hc, Wc = base_crop.shape

# ============================================================
# Crop masks
# ============================================================
true_crop = crop(true_masks, x0, x1, y0, y1)
cp_crop   = crop(cp_masks,   x0, x1, y0, y1)
xr_crop   = crop(xr_masks,   x0, x1, y0, y1)

# IMPORTANT: remove any instances touching the CROPPED ROI border
true_crop_clean = remove_instances_touching_crop_border(true_crop)
cp_crop_clean   = remove_instances_touching_crop_border(cp_crop)
xr_crop_clean   = remove_instances_touching_crop_border(xr_crop)

# ============================================================
# OUTLINES (from cleaned masks only)
# ============================================================
dilate_iters = 1

true_outline = binary_dilation(masks_to_outlines(true_crop_clean), iterations=dilate_iters)
cp_outline   = binary_dilation(masks_to_outlines(cp_crop_clean),   iterations=dilate_iters)
xr_outline   = binary_dilation(masks_to_outlines(xr_crop_clean),   iterations=dilate_iters)

# ============================================================
# Spots (crop + inside/outside) — use CLEAN masks for consistency
# ============================================================
spots_xy = spot["spots_xy_px"]
sx, sy = spots_xy[:, 0], spots_xy[:, 1]

in_roi = (sx >= x0) & (sx < x1) & (sy >= y0) & (sy < y1)
sx = (sx[in_roi] - x0).astype(int)
sy = (sy[in_roi] - y0).astype(int)

# safety guard
valid = (sx >= 0) & (sx < Wc) & (sy >= 0) & (sy < Hc)
sx, sy = sx[valid], sy[valid]

gt_in, gt_out = split_spots_by_mask(true_crop_clean, sx, sy)
cp_in, cp_out = split_spots_by_mask(cp_crop_clean,   sx, sy)
xr_in, xr_out = split_spots_by_mask(xr_crop_clean,   sx, sy)

# ============================================================
# Build panels
# ============================================================
rgb_gt = make_base_rgb(base_crop)
rgb_gt[true_outline] = [1, 0, 0]     # red

rgb_cp = make_base_rgb(base_crop)
rgb_cp[cp_outline] = [0, 1, 0]       # green

rgb_xr = make_base_rgb(base_crop)
rgb_xr[xr_outline] = [1, 0, 1]       # magenta

# ============================================================
# Plot
# ============================================================
fig, ax = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)

panels = [
    (rgb_gt, gt_in, gt_out, "Ground "),
    (rgb_cp, cp_in, cp_out, "HALO"),
    (rgb_xr, xr_in, xr_out, "XR"),
]

for a, (rgb, inside, outside, title) in zip(ax, panels):
    a.imshow(rgb)
    a.scatter(*inside, s=6, c="cyan", linewidths=0)
    a.scatter(*outside, s=6, c="yellow", linewidths=0)
    a.set_title(title)
    a.axis("off")

plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import binary_dilation
from cellpose.utils import masks_to_outlines

# ============================================================
# Settings (change these only)
# ============================================================
TARGET_CELL_ID = 94214

base_img = img[..., 0].astype(float)
mn, mx = base_img.min(), base_img.max()
if mx > mn:
    base_img = (base_img - mn) / (mx - mn)

# Crop option: center + size
cx, cy = 345, 230
roi_w, roi_h = 250, 250

# Outline thickness
CELL_THICKNESS = 1
NUC_THICKNESS  = 1
# ============================================================

# ------------------------------------------------------------
# Compute crop box from center + size
# ------------------------------------------------------------
H, W = base_img.shape
x0 = max(0, int(cx - roi_w // 2))
x1 = min(W, int(cx + roi_w // 2))
y0 = max(0, int(cy - roi_h // 2))
y1 = min(H, int(cy + roi_h // 2))

def crop(a):
    return a[y0:y1, x0:x1]

# ------------------------------------------------------------
# Base image (BLUE) in crop
# ------------------------------------------------------------
base_crop = crop(base_img)
Hc, Wc = base_crop.shape

def make_base_rgb():
    rgb = np.zeros((Hc, Wc, 3), dtype=float)
    rgb[..., 2] = base_crop  # BLUE channel
    return rgb

# ------------------------------------------------------------
# Keep ONLY the target cell in the cropped label mask
# ------------------------------------------------------------
cell_lab = crop(true_masks).astype(np.int32)

if not np.any(cell_lab == TARGET_CELL_ID):
    present_ids = np.unique(cell_lab)
    present_ids = present_ids[present_ids != 0]
    raise ValueError(
        f"Cell ID {TARGET_CELL_ID} not found in this crop. "
        f"Found {len(present_ids)} cell IDs in crop."
    )

cell_only = (cell_lab == TARGET_CELL_ID)

# ------------------------------------------------------------
# Keep ONLY nuclei pixels that fall inside this cell
#   (works whether nuclei_masks is binary or labeled)
# ------------------------------------------------------------
nuc_lab = crop(nuclei_masks)
nuc_only = (nuc_lab > 0) & cell_only

# ------------------------------------------------------------
# Outlines (Cellpose outlines + dilation)
# ------------------------------------------------------------
cell_outline = masks_to_outlines(cell_only.astype(np.int32))
cell_outline = binary_dilation(cell_outline, iterations=CELL_THICKNESS)

nuc_outline = masks_to_outlines(nuc_only.astype(np.int32))
nuc_outline = binary_dilation(nuc_outline, iterations=NUC_THICKNESS)

# ------------------------------------------------------------
# Build RGB (blue base + green nuclei outline + red cell outline)
# ------------------------------------------------------------
rgb = make_base_rgb()

# draw nuclei first, then cell on top (so cell boundary stays visible)
rgb[nuc_outline]  = [0, 1, 0]  # GREEN nuclei outline
rgb[cell_outline] = [1, 0, 0]  # RED cell outline

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------
plt.figure(figsize=(5, 5))
plt.imshow(rgb)
plt.title('')
plt.axis("off")
plt.show()


In [ ]:
tile_id = "tile_0008_base"

img = imread(TILES_DIR / "human_pancreas" / 'test' / 'images' / f"{tile_id}.tiff")
true_masks = np.load(TILES_DIR / "human_pancreas" / 'test' / 'masks' / f'{tile_id}.npy')
nuclei_masks = np.load(TILES_DIR / "human_pancreas" / 'test' / 'nuclei_mask' / f'{tile_id}.npy')
cp_masks =  np.load(TILES_DIR / "human_pancreas" / 'test' / 'mask_cpsam' / f'{tile_id}.npy')
xr_masks = np.load(TILES_DIR / 'human_pancreas' / 'test' / 'mask_xr' / f'{tile_id}.npy')
spot = np.load(TILES_DIR / 'human_pancreas' / 'test' / 'spots' / f'{tile_id}.npz')

true_outline = masks_to_outlines(true_masks)
cp_outline = masks_to_outlines(cp_masks)
xr_outline = masks_to_outlines(xr_masks)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import binary_dilation
from cellpose.utils import masks_to_outlines

# ============================================================
# Normalize channels
# ============================================================
# Base image (channel 0) → BLUE
base_img = img[..., 0].astype(float)
mn, mx = base_img.min(), base_img.max()
if mx > mn:
    base_img = (base_img - mn) / (mx - mn)

# Info image (channel 1) → PINK (red + blue)
info_img = img[..., 1].astype(float)
mn, mx = info_img.min(), info_img.max()
if mx > mn:
    info_img = (info_img - mn) / (mx - mn)

# ============================================================
# Crop config (ONLY change these)
# ============================================================
cx, cy = 256, 256
roi_w, roi_h = 512, 512
H, W = base_img.shape
x0 = max(0, int(cx - roi_w // 2))
x1 = min(W, int(cx + roi_w // 2))
y0 = max(0, int(cy - roi_h // 2))
y1 = min(H, int(cy + roi_h // 2))
# ============================================================

def crop(a):
    return a[y0:y1, x0:x1]

# ------------------------------------------------------------
# Cropped base & info
# ------------------------------------------------------------
base_crop = crop(base_img)
info_crop = crop(info_img)
Hc, Wc = base_crop.shape

def make_base_rgb():
    rgb = np.zeros((Hc, Wc, 3), dtype=float)
    rgb[..., 2] = base_crop                 # BLUE
    rgb[..., 0] += info_crop * 0.8          # RED   (pink)
    rgb[..., 2] += info_crop * 0.8          # BLUE  (pink)
    rgb = np.clip(rgb, 0, 1)
    return rgb

# ------------------------------------------------------------
# Label cell IDs (centroid-based)
# ------------------------------------------------------------
cell_lab = crop(true_masks).astype(np.int32)

cell_ids = np.unique(cell_lab)
cell_ids = cell_ids[cell_ids != 0]  # remove background

# optional: skip very small cells to avoid clutter
MIN_PIXELS = 50

centroids = []
for cid in cell_ids:
    ys, xs = np.where(cell_lab == cid)
    if ys.size < MIN_PIXELS:
        continue
    cy_i = ys.mean()
    cx_i = xs.mean()
    centroids.append((cid, cx_i, cy_i))


# ------------------------------------------------------------
# OUTLINES — EXACTLY your previous way
# ------------------------------------------------------------
nuc_outline  = masks_to_outlines(crop(nuclei_masks))
cell_outline = masks_to_outlines(crop(true_masks))

nuc_outline  = binary_dilation(nuc_outline, iterations=1)
cell_outline = binary_dilation(cell_outline, iterations=1)

# ------------------------------------------------------------
# Build RGB panels
# ------------------------------------------------------------
rgb_nuc = make_base_rgb()
rgb_nuc[nuc_outline] = [0, 1, 0]   # GREEN nuclei outline

rgb_cell = make_base_rgb()
rgb_cell[cell_outline] = [1, 0, 0] # RED cell outline

# ------------------------------------------------------------
# Horizontal 2 subplots
# ------------------------------------------------------------
# ------------------------------------------------------------
# Horizontal 2 subplots
# ------------------------------------------------------------
fig, ax = plt.subplots(1, 2, figsize=(10, 5), constrained_layout=True)

# --- nuclei panel ---
ax[0].imshow(rgb_nuc)
ax[0].set_title("")
ax[0].axis("off")

# --- cell panel ---
ax[1].imshow(rgb_cell)
ax[1].set_title("")
ax[1].axis("off")

# overlay cell IDs
for cid, cx_i, cy_i in centroids:
    ax[1].text(
        cx_i, cy_i, str(int(cid)),
        color="white",
        fontsize=8,
        ha="center",
        va="center",
        bbox=dict(
            boxstyle="round,pad=0.15",
            facecolor="black",
            alpha=0.6,
            linewidth=0
        )
    )

plt.show()




In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import binary_dilation
from cellpose.utils import masks_to_outlines

# ============================================================
# Target cell
# ============================================================
TARGET_CELL_ID = 83302

# ============================================================
# Normalize channels
# ============================================================
base_img = img[..., 0].astype(float)
mn, mx = base_img.min(), base_img.max()
if mx > mn:
    base_img = (base_img - mn) / (mx - mn)

info_img = img[..., 1].astype(float)
mn, mx = info_img.min(), info_img.max()
if mx > mn:
    info_img = (info_img - mn) / (mx - mn)

# ============================================================
# Crop config
# ============================================================
cx, cy = 450, 150
roi_w, roi_h = 150, 150
H, W = base_img.shape
x0 = max(0, int(cx - roi_w // 2))
x1 = min(W, int(cx + roi_w // 2))
y0 = max(0, int(cy - roi_h // 2))
y1 = min(H, int(cy + roi_h // 2))

def crop(a):
    return a[y0:y1, x0:x1]

# ------------------------------------------------------------
# Cropped images
# ------------------------------------------------------------
base_crop = crop(base_img)
info_crop = crop(info_img)
Hc, Wc = base_crop.shape

def make_base_rgb():
    rgb = np.zeros((Hc, Wc, 3), dtype=float)
    rgb[..., 2] = base_crop            # BLUE
    rgb[..., 0] += info_crop * 0.8     # RED  (pink)
    rgb[..., 2] += info_crop * 0.8     # BLUE (pink)
    return np.clip(rgb, 0, 1)

# ------------------------------------------------------------
# Keep ONLY the target cell
# ------------------------------------------------------------
cell_lab = crop(true_masks).astype(np.int32)

if not np.any(cell_lab == TARGET_CELL_ID):
    raise ValueError(f"Cell {TARGET_CELL_ID} not found in this crop.")

cell_only = (cell_lab == TARGET_CELL_ID)

# ------------------------------------------------------------
# Keep ONLY nuclei inside that cell
#   (robust to binary or labeled nuclei masks)
# ------------------------------------------------------------
nuc_lab = crop(nuclei_masks)
nuc_only = (nuc_lab > 0) & cell_only

# ------------------------------------------------------------
# Outlines — your original method
# ------------------------------------------------------------
cell_outline = masks_to_outlines(cell_only.astype(np.int32))
cell_outline = binary_dilation(cell_outline, iterations=1)

nuc_outline = masks_to_outlines(nuc_only.astype(np.int32))
nuc_outline = binary_dilation(nuc_outline, iterations=1)

# ------------------------------------------------------------
# Build RGB
# ------------------------------------------------------------
rgb = make_base_rgb()
rgb[nuc_outline]  = [0, 1, 0]  # GREEN nuclei outline
rgb[cell_outline] = [1, 0, 0]  # RED cell outline

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------
plt.figure(figsize=(5, 5))
plt.imshow(rgb)
plt.title('')
plt.axis("off")
plt.show()


In [ ]:
tile_id = "tile_0014_base"

img = imread(TILES_DIR / "human_pancreas" / 'train' / 'images' / f"{tile_id}.tiff")
true_masks = np.load(TILES_DIR / "human_pancreas" / 'train' / 'masks' / f'{tile_id}.npy')
nuclei_masks = np.load(TILES_DIR / "human_pancreas" / 'train' / 'nuclei_mask' / f'{tile_id}.npy')
cp_masks =  np.load(TILES_DIR / "human_pancreas" / 'train' / 'mask_cpsam' / f'{tile_id}.npy')
# xr_masks = np.load(TILES_DIR / 'human_pancreas' / 'train' / 'mask_xr' / f'{tile_id}.npy')
spot = np.load(TILES_DIR / 'human_pancreas' / 'train' / 'spots' / f'{tile_id}.npz')

spots_xy_px = spot['spots_xy_px']

# Optional ROI crop to focus on part of the tile.
# Set to None for full tile.
# Format: (x0, x1, y0, y1)

focus_roi = (100,250,350,500)
# focus_roi = None

if focus_roi is not None:
    x0, x1, y0, y1 = map(int, focus_roi)
    H, W = img.shape[:2]
    x0 = max(0, min(x0, W))
    x1 = max(0, min(x1, W))
    y0 = max(0, min(y0, H))
    y1 = max(0, min(y1, H))

    if x0 >= x1 or y0 >= y1:
        raise ValueError('focus_roi must define a non-empty region inside the tile.')

    img = img[y0:y1, x0:x1]
    true_masks = true_masks[y0:y1, x0:x1]
    nuclei_masks = nuclei_masks[y0:y1, x0:x1]
    cp_masks = cp_masks[y0:y1, x0:x1]
    xr_masks = xr_masks[y0:y1, x0:x1]

    in_roi = (
        (spots_xy_px[:, 0] >= x0) & (spots_xy_px[:, 0] < x1) &
        (spots_xy_px[:, 1] >= y0) & (spots_xy_px[:, 1] < y1)
    )
    spots_xy_px = spots_xy_px[in_roi].copy()
    spots_xy_px[:, 0] -= x0
    spots_xy_px[:, 1] -= y0

true_outline = masks_to_outlines(true_masks)
cp_outline = masks_to_outlines(cp_masks)
# xr_outline = masks_to_outlines(xr_masks)

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

black_blue = LinearSegmentedColormap.from_list(
    "black_blue",
    [(0, 0, 0), (0, 0, 1)]
)

black_pink = LinearSegmentedColormap.from_list(
    "black_pink",
    [(0, 0, 0), (1, 0, 1)]
)

plt.imshow(img[..., 0], cmap=black_blue)
plt.axis('off')


In [ ]:
plt.imshow(img[...,1],cmap='inferno')
plt.axis('off')


In [ ]:
# Optional extra zoom for channel view (on top of focus_roi).
# Set to None to use the full current img.
# Format: (x0, x1, y0, y1)
channel_view_roi = None

if channel_view_roi is None:
    ch0 = img[..., 0]
    ch1 = img[..., 1]
else:
    x0, x1, y0, y1 = map(int, channel_view_roi)
    ch0 = img[y0:y1, x0:x1, 0]
    ch1 = img[y0:y1, x0:x1, 1]

if ch0.size == 0 or ch1.size == 0:
    raise ValueError('channel_view_roi produced an empty crop. Check roi bounds.')

def norm(x):
    mn, mx = x.min(), x.max()
    return (x - mn) / (mx - mn) if mx > mn else x

ch0 = norm(ch0)
ch1 = norm(ch1)

# --- build RGB ---
rgb = np.zeros((*ch0.shape, 3), dtype=float)

# channel 0 → BLUE
rgb[..., 2] += ch0

# channel 1 → PINK = RED + BLUE
rgb[..., 0] += ch1
rgb[..., 2] += ch1

# clip to [0, 1]
rgb = np.clip(rgb, 0, 1)

# --- show ---
plt.imshow(rgb)
plt.axis('off')
plt.show()


In [ ]:
spots_loc = spots_xy_px
spots_loc

In [ ]:
H, W = img.shape[:2]
mask = np.zeros((H, W), dtype=np.uint8)

x = spots_loc[:, 0].astype(int)
y = spots_loc[:, 1].astype(int)

valid = (x >= x0) & (x < x1) & (y >= y0) & (y < y1)
x = x[valid] - x0
y = y[valid] - y0

mask[y, x] = 1
# mask[np.clip(y-1, 0, H-1), x] = 1
# mask[np.clip(y+1, 0, H-1), x] = 1
# mask[y, np.clip(x-1, 0, W-1)] = 1
# mask[y, np.clip(x+1, 0, W-1)] = 1

yellow_mask = np.zeros((H, W, 3), dtype=float)
yellow_mask[mask > 0] = [1.0, 1.0, 0.0]  # yellow
plt.imshow(yellow_mask)
plt.axis('off')

In [ ]:
from scipy.ndimage import gaussian_filter

cell_outline = masks_to_outlines(true_masks).astype(float)

# Smooth the boundary by blurring the binary outline.
smooth_outline = gaussian_filter(cell_outline, sigma=1.0)
smooth_outline = np.clip(smooth_outline * 2.5, 0, 1)

outline = np.zeros((*smooth_outline.shape, 3), dtype=float)
outline[..., 1] = smooth_outline  # green channel

plt.imshow(outline, interpolation='bilinear')
plt.axis('off')